# Kaggle ASTGCN 与 baseline 性能对比

本 notebook 面向 Kaggle/Ubuntu 环境，用于自动定位项目代码和 PEMS04 数据文件，复用 `src/astgcn` 包与 `scripts/compare_baselines.py`，比较 `HA`、`SVR`、`LSTM`、`GRU`、`ASTGCN` 的 MAE、RMSE、MAPE，并生成指标表和预测曲线。

默认使用 `quick` 模式做快速连通性检查；正式实验时将 `RUN_MODE` 改为 `full`。

In [ ]:
!git clone https://github.com/Tuzfucius/ASTGCN-learning

from pathlib import Path
import os
import subprocess
import sys


def find_project_root():
    candidates = [Path.cwd(), Path('/kaggle/working/ASTGCN'), Path('/kaggle/working')]
    for start in candidates:
        if not start.exists():
            continue
        for path in [start, *start.parents]:
            if (path / 'pyproject.toml').exists() and (path / 'src' / 'astgcn').exists():
                return path
        for child in start.glob('**/pyproject.toml'):
            root = child.parent
            if (root / 'src' / 'astgcn').exists():
                return root
    raise FileNotFoundError('未找到 ASTGCN 项目根目录')


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('PROJECT_ROOT =', PROJECT_ROOT)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'], check=True)


In [ ]:
import json
import subprocess
import sys
from copy import deepcopy

import matplotlib
import matplotlib.font_manager as font_manager
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

from astgcn.data.dataloader import build_dataloaders
from astgcn.data.io import load_pems_npz


def setup_chinese_font():
    candidate_names = [
        'Noto Sans CJK SC',
        'Noto Sans CJK JP',
        'Microsoft YaHei',
        'SimHei',
        'WenQuanYi Micro Hei',
        'Arial Unicode MS',
    ]
    for name in candidate_names:
        try:
            font_manager.findfont(name, fallback_to_default=False)
            plt.rcParams['font.family'] = 'sans-serif'
            plt.rcParams['font.sans-serif'] = [name, 'DejaVu Sans']
            plt.rcParams['axes.unicode_minus'] = False
            print('Matplotlib 中文字体 =', name)
            return name
        except ValueError:
            pass

    if sys.platform.startswith('linux'):
        print('未发现中文字体，尝试安装 fonts-noto-cjk。')
        subprocess.run(['apt-get', 'update', '-qq'], check=False)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-noto-cjk'], check=False)
        font_paths = []
        for root in ['/usr/share/fonts/opentype/noto', '/usr/share/fonts/truetype/noto']:
            font_paths.extend(Path(root).glob('NotoSansCJK*.ttc'))
            font_paths.extend(Path(root).glob('NotoSansCJK*.otf'))
        if font_paths:
            font_manager.fontManager.addfont(str(font_paths[0]))
            font_name = font_manager.FontProperties(fname=str(font_paths[0])).get_name()
            plt.rcParams['font.family'] = 'sans-serif'
            plt.rcParams['font.sans-serif'] = [font_name, 'DejaVu Sans']
            plt.rcParams['axes.unicode_minus'] = False
            print('Matplotlib 中文字体 =', font_name)
            return font_name

    plt.rcParams['axes.unicode_minus'] = False
    print('警告：未找到中文字体，图表中文可能仍显示为方块。')
    return None


CHINESE_FONT = setup_chinese_font()


## 运行配置

- `quick`：用于 Kaggle 快速调试，只跑少量 batch，并限制 SVR 样本数。
- `full`：使用更多 batch 和配置中的 epoch 数，适合正式对比实验。

In [ ]:
RUN_MODE = 'quick'  # 正式实验可改为 'full'

with open(PROJECT_ROOT / 'configs' / 'pems04.yaml', 'r', encoding='utf-8') as file:
    cfg = yaml.safe_load(file)


def find_existing_file(names):
    search_roots = [
        PROJECT_ROOT / 'data' / 'raw' / 'PEMS04',
        PROJECT_ROOT / 'data' / 'PEMS04',
        Path('/kaggle/input'),
        Path('/kaggle/working'),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        for name in names:
            direct = root / name
            if direct.exists():
                return direct
        for name in names:
            matches = list(root.glob(f'**/{name}'))
            if matches:
                return matches[0]
    raise FileNotFoundError(f'未找到数据文件: {names}')


data_path = find_existing_file(['pems04.npz', 'PEMS04.npz'])
distance_path = find_existing_file(['distance.csv'])
print('data_path =', data_path)
print('distance_path =', distance_path)

run_cfg = deepcopy(cfg)
run_cfg['dataset']['data_path'] = str(data_path)
run_cfg['dataset']['distance_path'] = str(distance_path)
run_cfg['train']['device'] = 'auto'

if RUN_MODE == 'quick':
    epochs = 1
    max_batches = 1
    svr_samples = 128
else:
    epochs = int(run_cfg['train']['epochs'])
    max_batches = 20
    svr_samples = 2048

comparison_dir = PROJECT_ROOT / 'outputs' / 'comparison'
comparison_dir.mkdir(parents=True, exist_ok=True)
CONFIG_FOR_RUN = comparison_dir / 'kaggle_pems04.yaml'
with open(CONFIG_FOR_RUN, 'w', encoding='utf-8') as file:
    yaml.safe_dump(run_cfg, file, allow_unicode=True, sort_keys=False)
print('CONFIG_FOR_RUN =', CONFIG_FOR_RUN)
print('epochs =', epochs, 'max_batches =', max_batches, 'svr_samples =', svr_samples)


## 数据集与时间窗口

先查看 PEMS04 的原始数据形状、训练/验证/测试时间范围，以及 ASTGCN 实际使用的 recent、daily、weekly 三类输入窗口。

In [ ]:
raw_data = load_pems_npz(data_path)
dataset_cfg = run_cfg['dataset']
window_cfg = run_cfg['time_window']
train_cfg = run_cfg['train']
split_cfg = run_cfg['split']
target_dim = int(dataset_cfg['target_dim'])

train_loader, val_loader, test_loader, scaler = build_dataloaders(
    data_path=dataset_cfg['data_path'],
    num_recent=window_cfg['recent_len'],
    num_days=window_cfg['daily_days'],
    num_weeks=window_cfg['weekly_weeks'],
    pred_len=window_cfg['pred_len'],
    batch_size=train_cfg['batch_size'],
    points_per_day=dataset_cfg['points_per_day'],
    target_dim=target_dim,
    train_ratio=split_cfg['train_ratio'],
    val_ratio=split_cfg['val_ratio'],
    num_workers=train_cfg.get('num_workers', 0),
)
train_dataset = train_loader.dataset
val_dataset = val_loader.dataset
test_dataset = test_loader.dataset

data_summary = pd.DataFrame([
    {'item': '原始时间步 T', 'value': raw_data.shape[0]},
    {'item': '传感器节点 N', 'value': raw_data.shape[1]},
    {'item': '特征维度 F', 'value': raw_data.shape[2]},
    {'item': '目标特征 target_dim', 'value': target_dim},
    {'item': 'recent 输入步数', 'value': window_cfg['recent_len']},
    {'item': 'daily 输入步数', 'value': window_cfg['daily_days'] * window_cfg['pred_len']},
    {'item': 'weekly 输入步数', 'value': window_cfg['weekly_weeks'] * window_cfg['pred_len']},
    {'item': '预测步数', 'value': window_cfg['pred_len']},
])
display(data_summary)

split_summary = pd.DataFrame([
    {'split': 'train', 'samples': len(train_dataset), 't0_start': int(train_dataset.t0_list[0]), 't0_end': int(train_dataset.t0_list[-1])},
    {'split': 'val', 'samples': len(val_dataset), 't0_start': int(val_dataset.t0_list[0]), 't0_end': int(val_dataset.t0_list[-1])},
    {'split': 'test', 'samples': len(test_dataset), 't0_start': int(test_dataset.t0_list[0]), 't0_end': int(test_dataset.t0_list[-1])},
])
display(split_summary)
print('target mean =', float(scaler.mean[0, 0, target_dim]), 'target std =', float(scaler.std[0, 0, target_dim]))


## 训练样本序列可视化

下面选择一个训练样本和一个节点，展示 recent、daily、weekly 输入片段，以及模型需要预测的未来目标序列。

In [ ]:
DISPLAY_SAMPLE_ID = 0
DISPLAY_NODE_ID = 0
MAX_HEATMAP_NODES = 40

sample = train_dataset[DISPLAY_SAMPLE_ID]


def inverse_target(values):
    return scaler.inverse_transform_target(np.asarray(values), target_dim=target_dim)


recent_seq = inverse_target(sample['recent'][DISPLAY_NODE_ID, target_dim].numpy())
daily_seq = inverse_target(sample['daily'][DISPLAY_NODE_ID, target_dim].numpy())
weekly_seq = inverse_target(sample['weekly'][DISPLAY_NODE_ID, target_dim].numpy())
target_seq = inverse_target(sample['target'][DISPLAY_NODE_ID].numpy())

fig, axes = plt.subplots(2, 2, figsize=(13, 7), constrained_layout=True)
series = [
    ('recent 输入', recent_seq, axes[0, 0], '#4c78a8'),
    ('daily 输入', daily_seq, axes[0, 1], '#f58518'),
    ('weekly 输入', weekly_seq, axes[1, 0], '#54a24b'),
    ('未来 target', target_seq, axes[1, 1], '#111111'),
]
for title, values, axis, color in series:
    axis.plot(np.arange(1, len(values) + 1), values, marker='o', color=color, linewidth=1.8)
    axis.set_title(title)
    axis.set_xlabel('窗口内时间步')
    axis.set_ylabel('Traffic flow')
    axis.grid(alpha=0.25)
fig.suptitle(f'训练样本序列: sample={DISPLAY_SAMPLE_ID}, node={DISPLAY_NODE_ID}, t0={int(sample["t0"])}')
fig.savefig(comparison_dir / 'notebook_training_sample_sequences.png', dpi=160)
plt.show()

recent_heatmap = inverse_target(sample['recent'][:MAX_HEATMAP_NODES, target_dim, :].numpy())
fig, axis = plt.subplots(figsize=(11, 6), constrained_layout=True)
image = axis.imshow(recent_heatmap, aspect='auto', cmap='viridis')
axis.set_title(f'recent 输入热力图: 前 {recent_heatmap.shape[0]} 个节点')
axis.set_xlabel('recent 时间步')
axis.set_ylabel('节点编号')
fig.colorbar(image, ax=axis, label='Traffic flow')
fig.savefig(comparison_dir / 'notebook_recent_heatmap.png', dpi=160)
plt.show()


## 训练与性能对比

运行统一对比脚本，训练或评估 `HA`、`SVR`、`LSTM`、`GRU`、`ASTGCN`，并将结果保存到 `outputs/comparison/`。

In [ ]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'compare_baselines.py'),
    '--config', str(CONFIG_FOR_RUN),
    '--epochs', str(epochs),
    '--max-batches', str(max_batches),
    '--svr-samples', str(svr_samples),
    '--device', run_cfg['train']['device'],
    '--output-dir', str(PROJECT_ROOT / 'outputs' / 'comparison'),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 原始指标结果

`baseline_metrics.csv` 保留统一对比脚本输出的原始 MAE、RMSE、MAPE。交通流数据中可能存在接近 0 的真实值，原始 MAPE 容易被极小分母放大，因此后续报告应优先使用 WAPE、sMAPE 和 Masked MAPE。

In [ ]:
comparison_dir = PROJECT_ROOT / 'outputs' / 'comparison'
metrics_path = comparison_dir / 'baseline_metrics.csv'
metrics = pd.read_csv(metrics_path)
display(metrics)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
colors = ['#4c78a8', '#f58518', '#54a24b', '#e45756', '#72b7b2']
for axis, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    axis.bar(metrics['model'], metrics[metric], color=colors[:len(metrics)])
    axis.set_title(f'原始 {metric}')
    axis.set_ylabel(metric)
    axis.grid(axis='y', alpha=0.25)
    axis.tick_params(axis='x', rotation=30)
plt.show()

## 预测曲线与误差曲线

选择一个测试样本和一个节点，对比所有模型的预测序列、真实序列，以及逐预测步残差。

In [ ]:
pred_file = comparison_dir / 'baseline_predictions.npz'
arrays = np.load(pred_file)
target = arrays['target']
model_names = [name for name in arrays.files if name != 'target']

POINTS_PER_STEP_MINUTES = 5
steps = np.arange(1, target.shape[-1] + 1)
horizon_minutes = steps * POINTS_PER_STEP_MINUTES

PLOT_SAMPLE_ID = 0
PLOT_NODE_ID = 0

fig, axis = plt.subplots(figsize=(11, 5), constrained_layout=True)
axis.plot(horizon_minutes, target[PLOT_SAMPLE_ID, PLOT_NODE_ID], marker='o', linewidth=2.4, color='#111111', label='Target')
for name in model_names:
    axis.plot(horizon_minutes, arrays[name][PLOT_SAMPLE_ID, PLOT_NODE_ID], marker='o', linewidth=1.5, label=name)
axis.set_title(f'预测曲线对比: sample={PLOT_SAMPLE_ID}, node={PLOT_NODE_ID}')
axis.set_xlabel('预测时长 / min')
axis.set_ylabel('Traffic flow')
axis.grid(alpha=0.25)
axis.legend(ncol=2)
fig.savefig(comparison_dir / 'notebook_prediction_lines.png', dpi=160)
plt.show()

fig, axis = plt.subplots(figsize=(11, 5), constrained_layout=True)
for name in model_names:
    residual = arrays[name][PLOT_SAMPLE_ID, PLOT_NODE_ID] - target[PLOT_SAMPLE_ID, PLOT_NODE_ID]
    axis.plot(horizon_minutes, residual, marker='o', linewidth=1.5, label=name)
axis.axhline(0.0, color='#111111', linewidth=1.2, linestyle='--')
axis.set_title(f'预测残差: prediction - target, sample={PLOT_SAMPLE_ID}, node={PLOT_NODE_ID}')
axis.set_xlabel('预测时长 / min')
axis.set_ylabel('Residual')
axis.grid(alpha=0.25)
axis.legend(ncol=2)
fig.savefig(comparison_dir / 'notebook_residual_lines.png', dpi=160)
plt.show()

## 稳定指标与相对提升率

原始 MAPE 在真实值接近 0 时会异常放大。这里补充 WAPE、sMAPE 和 Masked MAPE，并计算 ASTGCN 相比各 baseline 的相对下降比例，作为报告中的主要指标表。

In [ ]:
def mae(pred, true):
    return float(np.mean(np.abs(pred - true)))


def rmse(pred, true):
    return float(np.sqrt(np.mean((pred - true) ** 2)))


def wape(pred, true, eps=1e-6):
    return float(np.sum(np.abs(pred - true)) / (np.sum(np.abs(true)) + eps) * 100)


def smape(pred, true, eps=1e-6):
    return float(np.mean(2 * np.abs(pred - true) / (np.abs(pred) + np.abs(true) + eps)) * 100)


def masked_mape(pred, true, threshold=10.0, eps=1e-6):
    mask = np.abs(true) > threshold
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((pred[mask] - true[mask]) / (true[mask] + eps))) * 100)


metric_records = []
for name in model_names:
    pred = arrays[name]
    metric_records.append({
        'model': name,
        'MAE': mae(pred, target),
        'RMSE': rmse(pred, target),
        'WAPE(%)': wape(pred, target),
        'sMAPE(%)': smape(pred, target),
        'Masked_MAPE(%)': masked_mape(pred, target, threshold=10.0),
    })

stable_metrics = pd.DataFrame(metric_records)
display(stable_metrics)
stable_metrics.to_csv(comparison_dir / 'notebook_stable_metrics.csv', index=False, encoding='utf-8-sig')

In [ ]:
main_model = 'ASTGCN'
metrics_for_improve = ['MAE', 'RMSE', 'WAPE(%)', 'sMAPE(%)']

if main_model not in set(stable_metrics['model']):
    print(f'未找到 {main_model}，跳过相对提升率计算。')
    improve_df = pd.DataFrame()
else:
    ast_row = stable_metrics.loc[stable_metrics['model'] == main_model].iloc[0]
    improve_records = []
    for base_model in stable_metrics['model']:
        if base_model == main_model:
            continue
        base_row = stable_metrics.loc[stable_metrics['model'] == base_model].iloc[0]
        row = {'baseline': base_model}
        for metric in metrics_for_improve:
            base_value = float(base_row[metric])
            ast_value = float(ast_row[metric])
            row[f'{metric}_improvement(%)'] = (base_value - ast_value) / base_value * 100 if base_value != 0 else np.nan
        improve_records.append(row)
    improve_df = pd.DataFrame(improve_records)
    display(improve_df)
    improve_df.to_csv(comparison_dir / 'notebook_astgcn_improvement.csv', index=False, encoding='utf-8-sig')

## 分预测时长误差

统计所有测试样本和所有节点在每个未来预测时长上的 MAE、RMSE 和 WAPE。横坐标使用分钟，便于直接对应 5 分钟粒度的交通预测报告。

In [ ]:
horizon_records = []
for name in model_names:
    pred = arrays[name]
    for horizon_index in range(target.shape[-1]):
        pred_slice = pred[:, :, horizon_index]
        target_slice = target[:, :, horizon_index]
        horizon_records.append({
            'model': name,
            'horizon_min': int((horizon_index + 1) * POINTS_PER_STEP_MINUTES),
            'MAE': mae(pred_slice, target_slice),
            'RMSE': rmse(pred_slice, target_slice),
            'WAPE(%)': wape(pred_slice, target_slice),
        })

horizon_metrics = pd.DataFrame(horizon_records)
display(horizon_metrics)

fig, axis = plt.subplots(figsize=(11, 5), constrained_layout=True)
for name in model_names:
    sub = horizon_metrics[horizon_metrics['model'] == name]
    axis.plot(sub['horizon_min'], sub['MAE'], marker='o', linewidth=1.7, label=name)
axis.set_title('不同预测时长下的 MAE')
axis.set_xlabel('预测时长 / min')
axis.set_ylabel('MAE')
axis.grid(alpha=0.25)
axis.legend(ncol=2)
fig.savefig(comparison_dir / 'notebook_horizon_mae_minutes.png', dpi=160)
plt.show()

horizon_metrics.to_csv(comparison_dir / 'notebook_horizon_metrics_minutes.csv', index=False, encoding='utf-8-sig')

## 误差分布与预测-真实散点图

误差分布用于比较模型稳定性；预测-真实散点图用于观察模型是否存在系统性高估或低估。

In [ ]:
error_frames = []
for name in model_names:
    abs_error = np.abs(arrays[name] - target).reshape(-1)
    error_frames.append(pd.DataFrame({'model': name, 'absolute_error': abs_error}))
error_frame = pd.concat(error_frames, ignore_index=True)

fig, axis = plt.subplots(figsize=(11, 5), constrained_layout=True)
box_values = [error_frame.loc[error_frame['model'] == name, 'absolute_error'].values for name in model_names]
axis.boxplot(box_values, showfliers=False)
axis.set_xticks(np.arange(1, len(model_names) + 1))
axis.set_xticklabels(model_names)
axis.set_title('各模型绝对误差分布')
axis.set_xlabel('模型')
axis.set_ylabel('Absolute error')
axis.grid(axis='y', alpha=0.25)
fig.savefig(comparison_dir / 'notebook_error_distribution.png', dpi=160)
plt.show()

rng = np.random.default_rng(42)
flat_target = target.reshape(-1)
sample_count = min(3000, flat_target.size)
sample_index = rng.choice(flat_target.size, size=sample_count, replace=False)
cols = min(3, len(model_names))
rows = int(np.ceil(len(model_names) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4.6 * cols, 4.2 * rows), constrained_layout=True)
axes = np.asarray(axes).reshape(-1)
target_sample = flat_target[sample_index]
lower = float(np.min(target_sample))
upper = float(np.max(target_sample))
for axis, name in zip(axes, model_names):
    pred_sample = arrays[name].reshape(-1)[sample_index]
    axis.scatter(target_sample, pred_sample, s=8, alpha=0.35)
    axis.plot([lower, upper], [lower, upper], color='#111111', linestyle='--', linewidth=1.1)
    axis.set_title(name)
    axis.set_xlabel('Target')
    axis.set_ylabel('Prediction')
    axis.grid(alpha=0.2)
for axis in axes[len(model_names):]:
    axis.axis('off')
fig.suptitle('预测值与真实值散点图')
fig.savefig(comparison_dir / 'notebook_pred_vs_target.png', dpi=160)
plt.show()


## 节点级误差分析

按传感器节点统计 MAE，用于观察哪些路段更难预测。如果少数节点误差长期偏高，通常意味着这些路段流量波动更强，或当前图结构没有充分刻画其空间关联。

In [ ]:
node_mae = pd.DataFrame({
    name: np.mean(np.abs(arrays[name] - target), axis=(0, 2))
    for name in model_names
})
node_mae.index.name = 'node_id'

display(node_mae.describe().T)

fig, axis = plt.subplots(figsize=(12, 5), constrained_layout=True)
for name in model_names:
    axis.plot(node_mae.index, node_mae[name], linewidth=1.2, label=name)
axis.set_title('不同节点上的 MAE 分布')
axis.set_xlabel('节点编号')
axis.set_ylabel('MAE')
axis.grid(alpha=0.25)
axis.legend(ncol=2)
fig.savefig(comparison_dir / 'notebook_node_mae_lines.png', dpi=160)
plt.show()

node_mae.to_csv(comparison_dir / 'notebook_node_mae.csv', encoding='utf-8-sig')

if 'ASTGCN' in node_mae.columns:
    worst_nodes = node_mae['ASTGCN'].sort_values(ascending=False).head(10)
    display(worst_nodes)
    worst_nodes.to_csv(comparison_dir / 'notebook_astgcn_worst_nodes.csv', encoding='utf-8-sig')

## 节点-预测时长误差热力图

热力图展示 ASTGCN 的误差是否集中在远期预测，或集中在少数节点上。该图比整体平均误差更适合解释模型仍然失败的区域。

In [ ]:
MODEL_TO_ANALYZE = 'ASTGCN' if 'ASTGCN' in model_names else model_names[-1]
node_horizon_mae = np.mean(np.abs(arrays[MODEL_TO_ANALYZE] - target), axis=0)

fig, axis = plt.subplots(figsize=(10, 7), constrained_layout=True)
image = axis.imshow(node_horizon_mae, aspect='auto', cmap='viridis')
axis.set_title(f'{MODEL_TO_ANALYZE} 节点-预测时长 MAE 热力图')
axis.set_xlabel('预测时长')
axis.set_ylabel('节点编号')
axis.set_xticks(np.arange(target.shape[-1]))
axis.set_xticklabels([f'{POINTS_PER_STEP_MINUTES * (i + 1)}min' for i in range(target.shape[-1])], rotation=45)
fig.colorbar(image, ax=axis, label='MAE')
fig.savefig(comparison_dir / 'notebook_astgcn_node_horizon_heatmap.png', dpi=160)
plt.show()

pd.DataFrame(
    node_horizon_mae,
    columns=[f'{POINTS_PER_STEP_MINUTES * (i + 1)}min' for i in range(target.shape[-1])],
).to_csv(comparison_dir / 'notebook_astgcn_node_horizon_mae.csv', index_label='node_id', encoding='utf-8-sig')

## 高峰与非高峰时段误差

交通预测需要关注早晚高峰。这里根据测试样本的 `t0` 位置划分高峰和非高峰，比较不同模型在两个时段的平均样本 MAE。

In [ ]:
points_per_day = int(dataset_cfg['points_per_day'])
test_t0 = np.asarray(test_dataset.t0_list[:target.shape[0]])
hour_of_day = (test_t0 % points_per_day) * 24 / points_per_day
period = np.where(
    ((hour_of_day >= 7) & (hour_of_day < 9)) | ((hour_of_day >= 17) & (hour_of_day < 19)),
    'peak',
    'off_peak',
)

period_records = []
for name in model_names:
    sample_mae = np.mean(np.abs(arrays[name] - target), axis=(1, 2))
    for period_name in ['peak', 'off_peak']:
        mask = period == period_name
        period_records.append({
            'model': name,
            'period': period_name,
            'sample_count': int(mask.sum()),
            'MAE': float(sample_mae[mask].mean()) if mask.sum() > 0 else np.nan,
        })

period_df = pd.DataFrame(period_records)
display(period_df)

x = np.arange(len(model_names))
fig, axis = plt.subplots(figsize=(9, 5), constrained_layout=True)
for period_name, offset in [('peak', 0.18), ('off_peak', -0.18)]:
    sub = period_df[period_df['period'] == period_name].set_index('model').reindex(model_names)
    axis.bar(x + offset, sub['MAE'], width=0.35, label=period_name)
axis.set_xticks(x)
axis.set_xticklabels(model_names, rotation=30)
axis.set_title('高峰与非高峰时段 MAE 对比')
axis.set_ylabel('MAE')
axis.grid(axis='y', alpha=0.25)
axis.legend()
fig.savefig(comparison_dir / 'notebook_peak_offpeak_mae.png', dpi=160)
plt.show()

period_df.to_csv(comparison_dir / 'notebook_peak_offpeak_mae.csv', index=False, encoding='utf-8-sig')

## 最好、中等与最差案例

随机展示单个样本容易带来选择偏差。这里按 ASTGCN 的样本级 MAE 自动选择 best、median 和 worst 三类样本，并在每个样本中选择误差最大的节点，便于展示成功场景和失败场景。

In [ ]:
MODEL_TO_ANALYZE = 'ASTGCN' if 'ASTGCN' in model_names else model_names[-1]
sample_mae = np.mean(np.abs(arrays[MODEL_TO_ANALYZE] - target), axis=(1, 2))

case_indices = {
    'best': int(np.argmin(sample_mae)),
    'median': int(np.argsort(sample_mae)[len(sample_mae) // 2]),
    'worst': int(np.argmax(sample_mae)),
}
case_records = []

for case_name, sample_id in case_indices.items():
    node_error = np.mean(np.abs(arrays[MODEL_TO_ANALYZE][sample_id] - target[sample_id]), axis=1)
    node_id = int(np.argmax(node_error))
    case_records.append({
        'case': case_name,
        'sample_id': sample_id,
        'node_id': node_id,
        'sample_MAE': float(sample_mae[sample_id]),
        'node_MAE': float(node_error[node_id]),
    })

    fig, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
    axis.plot(horizon_minutes, target[sample_id, node_id], marker='o', linewidth=2.4, label='Target', color='#111111')
    axis.plot(horizon_minutes, arrays[MODEL_TO_ANALYZE][sample_id, node_id], marker='o', linewidth=1.8, label=MODEL_TO_ANALYZE)
    axis.set_title(f'{case_name} case: sample={sample_id}, node={node_id}, MAE={sample_mae[sample_id]:.2f}')
    axis.set_xlabel('预测时长 / min')
    axis.set_ylabel('Traffic flow')
    axis.grid(alpha=0.25)
    axis.legend()
    fig.savefig(comparison_dir / f'notebook_{MODEL_TO_ANALYZE}_{case_name}_case.png', dpi=160)
    plt.show()

case_df = pd.DataFrame(case_records)
display(case_df)
case_df.to_csv(comparison_dir / f'notebook_{MODEL_TO_ANALYZE}_case_summary.csv', index=False, encoding='utf-8-sig')

## 推理流程 shape 表

该表展示 DataLoader 输出、模型输入变换和最终预测数组的形状，用于报告中说明模型实际吃入什么数据、输出什么预测结果。

In [ ]:
batch = next(iter(test_loader))
shape_records = [
    {'stage': 'DataLoader recent', 'shape': str(tuple(batch['recent'].shape)), 'meaning': '[B, N, F, T_recent]'},
    {'stage': 'DataLoader daily', 'shape': str(tuple(batch['daily'].shape)), 'meaning': '[B, N, F, T_daily]'},
    {'stage': 'DataLoader weekly', 'shape': str(tuple(batch['weekly'].shape)), 'meaning': '[B, N, F, T_weekly]'},
    {'stage': 'Target', 'shape': str(tuple(batch['target'].shape)), 'meaning': '[B, N, Tp]'},
    {'stage': 'Model input recent', 'shape': str(tuple(batch['recent'].shape)), 'meaning': '[B, N, F, T_recent]'},
    {'stage': 'Model output', 'shape': str(target.shape), 'meaning': '[test_samples, N, Tp]'},
]

shape_df = pd.DataFrame(shape_records)
display(shape_df)
shape_df.to_csv(comparison_dir / 'notebook_inference_shape_flow.csv', index=False, encoding='utf-8-sig')

## ASTGCN 推理流程说明

对于每个预测时刻 `t0`，模型不会只使用一段连续历史，而是构造三类时间依赖输入：

1. `recent`：`t0` 前最近一段连续交通流，用于捕捉短期变化；
2. `daily`：前一天同一时段附近的交通流，用于捕捉日周期；
3. `weekly`：前一周同一时段附近的交通流，用于捕捉周周期。

每个分支输入形状为 `[B, N, F, T]`，其中 `B` 表示 batch size，`N` 表示传感器节点数，`F` 表示每个节点的特征数，`T` 表示历史时间窗口长度。

ASTGCN 分别对 `recent`、`daily`、`weekly` 三个分支进行时空注意力建模和图卷积建模，得到三个预测结果：`Y_recent`、`Y_daily`、`Y_weekly`。最后通过可学习融合权重进行逐元素加权，输出未来 `Tp` 个时间片的交通流预测：`Y_hat ∈ R^{B × N × Tp}`。

在 PEMS04 中，数据时间间隔为 5 分钟，因此 `Tp=12` 对应预测未来 60 分钟。

## 输出文件

本次运行会生成：

- `outputs/comparison/baseline_metrics.csv`
- `outputs/comparison/baseline_metrics.json`
- `outputs/comparison/metrics_bar.png`
- `outputs/comparison/sample_prediction.png`
- `outputs/comparison/baseline_predictions.npz`
- `outputs/comparison/notebook_training_sample_sequences.png`
- `outputs/comparison/notebook_recent_heatmap.png`
- `outputs/comparison/notebook_prediction_lines.png`
- `outputs/comparison/notebook_residual_lines.png`
- `outputs/comparison/notebook_stable_metrics.csv`
- `outputs/comparison/notebook_astgcn_improvement.csv`
- `outputs/comparison/notebook_horizon_mae_minutes.png`
- `outputs/comparison/notebook_horizon_metrics_minutes.csv`
- `outputs/comparison/notebook_error_distribution.png`
- `outputs/comparison/notebook_pred_vs_target.png`
- `outputs/comparison/notebook_node_mae_lines.png`
- `outputs/comparison/notebook_node_mae.csv`
- `outputs/comparison/notebook_astgcn_worst_nodes.csv`
- `outputs/comparison/notebook_astgcn_node_horizon_heatmap.png`
- `outputs/comparison/notebook_astgcn_node_horizon_mae.csv`
- `outputs/comparison/notebook_peak_offpeak_mae.png`
- `outputs/comparison/notebook_peak_offpeak_mae.csv`
- `outputs/comparison/notebook_ASTGCN_best_case.png`
- `outputs/comparison/notebook_ASTGCN_median_case.png`
- `outputs/comparison/notebook_ASTGCN_worst_case.png`
- `outputs/comparison/notebook_ASTGCN_case_summary.csv`
- `outputs/comparison/notebook_inference_shape_flow.csv`